# Task 4 — augmented game-title classification

Augmentation is applied only to training images. Test images use the same resize and feature extraction without augmentation.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageEnhance, ImageFile
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

ImageFile.LOAD_TRUNCATED_IMAGES = True
ROOT = Path.cwd()
DATA = ROOT / "data"
SUBMIT = ROOT / "submit"
RANDOM_STATE = 42

In [ ]:
def augment(image):
    # Conservative screenshot augmentation: no flip, so UI text remains readable.
    image = ImageEnhance.Brightness(image).enhance(0.90)
    image = ImageEnhance.Contrast(image).enhance(1.10)
    return image


def features(image):
    image = image.convert("RGB").resize((64, 32), Image.Resampling.BILINEAR)
    rgb = np.asarray(image, dtype=np.float32) / 255.0
    gray = rgb @ np.array([0.299, 0.587, 0.114], dtype=np.float32)
    pooled = rgb.reshape(8, 4, 8, 8, 3).mean(axis=(1, 3)).ravel()
    small = (
        np.asarray(
            image.resize((16, 8), Image.Resampling.BILINEAR), dtype=np.float32
        ).ravel()
        / 255.0
    )
    summary = np.concatenate(
        [
            rgb.mean(axis=(0, 1)),
            rgb.std(axis=(0, 1)),
            np.quantile(rgb, [0.1, 0.5, 0.9], axis=(0, 1)).ravel(),
        ]
    )
    edges = np.concatenate(
        [
            np.abs(np.diff(gray, axis=0)).mean(axis=0),
            np.abs(np.diff(gray, axis=1)).mean(axis=1),
        ]
    )
    return np.concatenate([pooled, small, summary, edges]).astype(np.float32)


def train_pair(path):
    with Image.open(path) as image:
        image = image.convert("RGB")
        return features(image), features(augment(image))


def test_features(path):
    with Image.open(path) as image:
        return features(image)

In [ ]:
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
train["class"] = train["class"].str.strip()
train_paths = [DATA / "train" / f"{x}.jpg" for x in train["id"]]
test_paths = [DATA / "test" / f"{x}.jpg" for x in test["id"]]
with ThreadPoolExecutor(max_workers=8) as pool:
    pairs = list(pool.map(train_pair, train_paths))
    X_test = np.vstack(list(pool.map(test_features, test_paths)))
X_original = np.vstack([p[0] for p in pairs])
X_augmented = np.vstack([p[1] for p in pairs])
X_train = np.vstack([X_original, X_augmented])
y_train = np.tile(train["class"].to_numpy(), 2)
print("augmented train:", X_train.shape, "test:", X_test.shape)

In [ ]:
tr, va = train_test_split(
    np.arange(len(y_train)), test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)
check = make_pipeline(
    StandardScaler(),
    SGDClassifier(
        loss="hinge",
        alpha=2e-5,
        class_weight="balanced",
        max_iter=40,
        tol=1e-3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
)
check.fit(X_train[tr], y_train[tr])
val_pred = check.predict(X_train[va])
print(
    "validation macro-F1:", round(f1_score(y_train[va], val_pred, average="macro"), 5)
)
print(classification_report(y_train[va], val_pred, zero_division=0))
model = make_pipeline(
    StandardScaler(),
    SGDClassifier(
        loss="hinge",
        alpha=2e-5,
        class_weight="balanced",
        max_iter=40,
        tol=1e-3,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
)
model.fit(X_train, y_train)
pred = model.predict(X_test)
pred = np.where(pred == "7-11k", "7-11K", pred)
sample = pd.read_csv(ROOT.parent / "submission" / "sample_submission.csv")
assert len(sample) == len(pred) == 25889
submission = pd.DataFrame({"id": sample["id"], "task4": pred})
submission.to_csv(SUBMIT / "index.csv", index=False)
print("wrote", SUBMIT / "index.csv")
print(submission["task4"].value_counts())